#### Forecasting Time series with ML models

We discuss and work with a few techniques such as time delay embedding and temporal embedding.

The approach we would have taken here is with first feature engineering, generating single-step forecast baselines, train and predict with ML models, standardize code to train and evaluate ML models.



##### Training and predicting with ML models

The goal with the supervised learning is to come up with a function, y_hat = h(X, phi) where y_hat is the predicted value, X is the set of features as the input and phi the model parameters, and h the approximation of the function. Here h is any function ~ the ideal function. and belonging to all possible functions from a family of functions H, and it is H that we call model. In linear regression for each values of the coefficients the model gives us a different function and H is the set of the possible functions.

What we discuss here is machine learning for forecasting (and later deep learning) and not general machine learning.






In [ ]:
import pandas as pd
import numpy as np
import os, pathlib
from pathlib import Path

import joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as io

from functools import partial

# Correcting the missing value handling methods in MLForecast
# This function replaces the deprecated `fillna(method=...)` with `bfill()` and `ffill()`
def handle_missing_values(df, bfill_columns, ffill_columns, zero_fill_columns):
    df_copy = df.copy()
    for col in bfill_columns:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].bfill()
    for col in ffill_columns:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].ffill()
    for col in zero_fill_columns:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].fillna(0)
    return df_copy

In [ ]:
import humanize
from sklearn.preprocessing import StandardScaler
from tqdm.autonotebook import tqdm
from IPython.display import display, HTML

# %load_ext autoreload
# %autoreload 2
np.random.seed(42)
import random

random.seed(42)
tqdm.pandas()

In [ ]:
import utilities

In [ ]:
exp_df = pd.read_parquet('partial_exp_df.parquet')

##### Feature Engineering for Machine Learning

In Feature Engineering for time series Forecasting the ML model can only predict one target at a time (see the single step forecast notebook) and generated baselines not just single step but multi-step. Generating single step forecast for baseline algorithms such as ARIMA or ETS requires to fit on history, predict one step ahead, and fit again using one more day. Predicting is iterative and it would require us to iterate for n-data points a day for 30 days, and repeating this for each household, which takes longer time to compute. Therefore we choose Naive and Seasonal Maine methods implemented as native pandas methods as two baseline methods to generate single-step forecasts.

Naive forecasts perform well for single-step-ahead forecasts and can be considered strong line.

We have generated the baseline for both validation and test datasets and saved the predictions, metrics and aggregated metrics to disk. To make training and evaluation easier we use a standard structure throughout.

#### Baseline

Train, Test and Validation Sets

In [ ]:
test_mask = (exp_block_df.timestamp.dt.year==2014) & \
            (exp_block_df.timestamp.dt.month==2)
validation_mask = (exp_block_df.timestamp.dt.year==2014) & \
            (exp_block_df.timestamp.dt.month==1)

train_df = exp_block_df[~(test_mask | validation_mask)]
validation_df = exp_block_df[validation_mask]
test_df = exp_block_df[test_mask]

train_df.shape, validation_df.shape, test_df.shape

((11855088, 20), (595200, 20), (518400, 20))

In [ ]:
'''
train_df = train_df[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
validation_df = validation_df[
    ['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
test_df = test_df[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
'''

In [ ]:
freq_ = train_df.iloc[0]['frequency']
timeseries_train = train_df.loc[
    train_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_validation = validation_df.loc[
    validation_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_test = test_df.loc[
    test_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]

In [ ]:
train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()

train_df["type"] = "train"
validation_df["type"] = "val"
test_df["type"] = "test"
full_df = pd.concat(
    [train_df, validation_df, test_df]).sort_values(["LCLid", "timestamp"])

In [ ]:
!pip install window_ops > /dev/null

Lag Features

In [ ]:
lags = ((np.arange(5) + 1).tolist() + (np.arange(5) + 46).tolist() +
        (np.arange(5) + (48 * 7) - 2).tolist() )
lags

[1, 2, 3, 4, 5, 46, 47, 48, 49, 50, 334, 335, 336, 337, 338]

In [ ]:
with LogTime():
    full_df, added_features = add_lags(
        full_df, lags = lags, column="energy_consumption", ts_id="LCLid", use_32_bit=True
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 10 seconds, 842 milliseconds and 399 microseconds
Features Created: energy_consumption_lag_1,energy_consumption_lag_2,energy_consumption_lag_3,energy_consumption_lag_4,energy_consumption_lag_5,energy_consumption_lag_46,energy_consumption_lag_47,energy_consumption_lag_48,energy_consumption_lag_49,energy_consumption_lag_50,energy_consumption_lag_334,energy_consumption_lag_335,energy_consumption_lag_336,energy_consumption_lag_337,energy_consumption_lag_338


Rolling

In [ ]:
with LogTime():
    full_df, added_features = add_rolling_features(
        full_df,
        rolls=[3, 6, 12, 48],
        column="energy_consumption",
        agg_funcs=["mean", "std"],
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 22 seconds, 994 milliseconds and 428 microseconds
Features Created: energy_consumption_rolling_3_mean,energy_consumption_rolling_3_std,energy_consumption_rolling_6_mean,energy_consumption_rolling_6_std,energy_consumption_rolling_12_mean,energy_consumption_rolling_12_std,energy_consumption_rolling_48_mean,energy_consumption_rolling_48_std


Seasonal Rolling

In [ ]:
with LogTime():
    full_df, added_features = add_seasonal_rolling_features(
        full_df,
        rolls=[3],
        seasonal_periods=[48, 48 * 7],
        column="energy_consumption",
        agg_funcs=["mean", "std"],
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 10 seconds, 651 milliseconds and 628 microseconds
Features Created: energy_consumption_48_seasonal_rolling_3_mean,energy_consumption_48_seasonal_rolling_3_std,energy_consumption_336_seasonal_rolling_3_mean,energy_consumption_336_seasonal_rolling_3_std


EWMA

In [ ]:
import math

t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
for alpha in [0.3, 0.5, 0.8]:
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    span = (2 - alpha) / alpha
    halflife = math.log(1 - alpha) / math.log(0.5)
    plot_df[f"Alpha={alpha} | Span={span:.2f}"] = weights

In [ ]:
fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
)
#fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
with LogTime():
    # full_df, added_features = add_ewma(full_df, alphas=[0.2, 0.5, 0.9], column="energy_consumption", ts_id="LCLid", use_32_bit=True)
    full_df, added_features = add_ewma(
        full_df,
        spans=[48 * 60, 48 * 7, 48],
        column="energy_consumption",
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 3 seconds, 489 milliseconds and 729 microseconds
Features Created: energy_consumption_ewma_span_2880,energy_consumption_ewma_span_336,energy_consumption_ewma_span_48


Temporal Features

In [ ]:
with LogTime():
    full_df, added_features = add_temporal_features(
        full_df,
        field_name="timestamp",
        frequency="30min",
        add_elapsed=True,
        drop=False,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 2 seconds, 761 milliseconds and 452 microseconds
Features Created: timestamp_Month,timestamp_Quarter,timestamp_Is_quarter_end,timestamp_Is_quarter_start,timestamp_Is_year_end,timestamp_Is_year_start,timestamp_Is_month_start,timestamp_Day,timestamp_Dayofweek,timestamp_Dayofyear,timestamp_Hour,timestamp_Minute,timestamp_Week,timestamp_Elapsed


Fourier Terms

In [ ]:
with LogTime():
    full_df, added_features = bulk_add_fourier_features(
        full_df,
        ["timestamp_Month", "timestamp_Hour", "timestamp_Minute"],
        max_values=[12, 24, 60],
        n_fourier_terms=5,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 7 seconds, 354 milliseconds and 195 microseconds
Features Created: timestamp_Month_sin_1,timestamp_Month_sin_2,timestamp_Month_sin_3,timestamp_Month_sin_4,timestamp_Month_sin_5,timestamp_Month_cos_1,timestamp_Month_cos_2,timestamp_Month_cos_3,timestamp_Month_cos_4,timestamp_Month_cos_5,timestamp_Hour_sin_1,timestamp_Hour_sin_2,timestamp_Hour_sin_3,timestamp_Hour_sin_4,timestamp_Hour_sin_5,timestamp_Hour_cos_1,timestamp_Hour_cos_2,timestamp_Hour_cos_3,timestamp_Hour_cos_4,timestamp_Hour_cos_5,timestamp_Minute_sin_1,timestamp_Minute_sin_2,timestamp_Minute_sin_3,timestamp_Minute_sin_4,timestamp_Minute_sin_5,timestamp_Minute_cos_1,timestamp_Minute_cos_2,timestamp_Minute_cos_3,timestamp_Minute_cos_4,timestamp_Minute_cos_5


In [ ]:
full_df.columns

Plotting Fourier Terms

In [ ]:
plot_df = (
    full_df[["timestamp_Month", "timestamp_Month_sin_1"]]
    .drop_duplicates()
    .sort_values("timestamp_Month")
)
plot_df.columns = ["calendar", "fourier"]

plot_df = pd.concat([plot_df, plot_df, plot_df]).reset_index(drop=True)
#plot_df.reset_index(drop=True, inplace=True)

plot_df.reset_index(inplace=True)
plot_df["index"] += 1
plot_df = pd.melt(
    plot_df, id_vars="index", var_name="month", value_name="Representation"
)

In [ ]:
fig = px.line(plot_df, x="index", y="Representation", facet_row="month")
fig.update_layout(
    autosize=False,
    width=900,
    height=800,
    title_text="Step Function vs Continuous Function",
    title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
    titlefont={"size": 20},
    legend_title=None,

    xaxis=dict(
        title_text="Time",
    ),
)
fig.update_yaxes(matches=None)
fig

In [ ]:
fig.update_xaxes(ticktext=np.arange(1, 13).tolist() * 3, tickvals=np.arange(len(plot_df)) + 1,)
fig

Saving the feature engineered file

In [ ]:
full_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 5000000 entries, 0 to 1519
Columns: 95 entries, timestamp to timestamp_Minute_cos_5
dtypes: datetime64[ns](1), float32(60), float64(8), int32(14), int64(2), object(10)
memory usage: 4.5 GB


In [ ]:
full_df[full_df["type"] == "train"].drop(columns="type").to_parquet("df_train_feature_eng.parquet")

full_df[full_df["type"] == "val"].drop(columns="type").to_parquet("df_val_feature_eng.parquet")

full_df[full_df["type"] == "test"].drop(columns="type").to_parquet("df_test_eature_eng.parquet")

In [ ]:
from google.colab import files
files.download('df_train_feature_eng.parquet')
files.download('df_val_feature_eng.parquet')
files.download('df_test_eature_eng.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Feature Engineering for Train and Test Dataset

Features we are creating need the train and test dataset to be combined into a single dataset with continuous time. In case of a production environment we create test period dataset with absent (zero) actual observations and continue.


In [ ]:
import collections
from collections import defaultdict

!pip install mlforecast > /dev/null
from mlforecast.lag_transforms import (
	RollingMean,
	RollingStd,
	RollingMin,
	RollingMax,
	SeasonalRollingMean,
	SeasonalRollingMin,
	SeasonalRollingMax,
	SeasonalRollingStd,
	ExponentiallyWeightedMean,
)

#### Lag Features

In [ ]:
lags = ((np.arange(5) + 1).tolist() + (np.arange(5) + 46).tolist() + (np.arange(5) + (48 * 7) - 2).tolist() )
lags

[1, 2, 3, 4, 5, 46, 47, 48, 49, 50, 334, 335, 336, 337, 338]

#### Rolling

All the other features apart from lags are added as LagTransforms in mlforecast, this due to a leakage in data (with rolling meand and current time step to be avoided). In mlforecast the transformations are added as a dictionary. To add transformations as {'1': [Transform for rolling Mean, Transform for Exponential Mean]} it will move back one time step and calculate the transformations.


In [ ]:
lag_transforms = defaultdict(list)

# Adding Rolling Mean, Rolling Std, with an offset of one timestep
lag_transforms[1]+= [
    RollingMean(window_size=n) for n in [3, 6, 12, 48]] + [RollingStd(window_size=n) for n in [3, 6, 12, 48] ]

#### Seasonal Rolling

This goes back seasonal length and calculates mean, max etc in a window from that starting point. Understand how to configure seasonal length and window size.

In [ ]:
''' Understand how to configure seasonal length and window size '''
from mlforecast.lag_transforms import SeasonalRollingMax
from utilsforecast.data import generate_series
from mlforecast import MLForecast

# generate sequential data to verify seasonal rolling
data = generate_series(1,min_length=50, max_length=500, seed=42)
data['y'] = np.arange(1, len(data)+1)

season_length = 8
window_size=1

# Defining the Rolling window we want to test
seasonal_rolling_window = SeasonalRollingMax(
     window_size=window_size, season_length=season_length)

fcst = MLForecast(
    models=[],
    freq='D',
    lag_transforms={
    season_length: [seasonal_rolling_window],
    },
)
data_t = fcst.preprocess(data)
data_t.head()

,unique_id,ds,y,seasonal_rolling_max_lag8_season_length8_window_size1
8,0,2000-01-09,9.0,1.0
9,0,2000-01-10,10.0,2.0
10,0,2000-01-11,11.0,3.0
11,0,2000-01-12,12.0,4.0
12,0,2000-01-13,13.0,5.0


In [ ]:
# Adding Seasonal Rolling Mean, Seasonal Rolling Std, with an offset of seasonal period timestep
lag_transforms[48]+= [SeasonalRollingMean(season_length=48, window_size=3)] + \
                     [SeasonalRollingStd(season_length=48, window_size=3)]

lag_transforms[48 * 7]+= [
    SeasonalRollingMean(season_length=48 * 7, window_size=3)] + [SeasonalRollingStd(season_length=48 * 7, window_size=3)]

#### EWMA

In [ ]:
import math

t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
for alpha in [0.3, 0.5, 0.8]:
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    span = (2 - alpha) / alpha
    halflife = math.log(1 - alpha) / math.log(0.5)
    plot_df[f"Alpha={alpha} | Span={span:.2f}"] = weights

In [ ]:
fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
)
fig.update_layout(
    autosize=False,
    width=1200,
    height=500,
    yaxis=dict(
        title_text="Weights",
        titlefont=dict(size=15),
        tickfont=dict(size=15),
    ),
    xaxis=dict(
        titlefont=dict(size=15),
        tickfont=dict(size=15),
    ),
)
fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
# Adding Rolling Mean, Rolling Std, with an offset of one timestep
lag_transforms[1] += [
    ExponentiallyWeightedMean(alpha=alpha) for alpha in [0.2, 0.5, 0.9]]

#### Temporal Features

In [ ]:
# Define the features you need in the model
# these should either be strings (pandas date function) or functions that take date as an argument
temporal_features = [
    "month", "quarter", "is_quarter_end", "is_quarter_start", "is_year_end", "is_year_start", "is_month_start", "is_month_end", "week", "day", "dayofweek", "dayofyear", "hour", "minute", ]

In [ ]:
with LogTime():
    full_df, added_features = add_temporal_features(
        full_df,
        field_name="timestamp",
        frequency="30min",
        add_elapsed=True,
        drop=False,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

#### Calculating the Features

In [ ]:
from mlforecast import MLForecast

In [ ]:
''' Understand how to configure seasonal length and window size '''
from mlforecast.lag_transforms import SeasonalRollingMax

# generate sequential data to verify seasonal rolling
data = generate_series(1,min_length=50, max_length=500, seed=42)
data['y'] = np.arange(1, len(data)+1)

season_length = 8
window_size=1

# Defining the Rolling window we want to test
seasonal_rolling_window = SeasonalRollingMax(
     window_size=window_size, season_length=season_length)

fcst = MLForecast(
    models=[],
    freq='D',
    lag_transforms={
    season_length: [seasonal_rolling_window],
    },
)
data_t = fcst.preprocess(data)
data_t.head()

,unique_id,ds,y,seasonal_rolling_max_lag8_season_length8_window_size1
8,0,2000-01-09,9.0,1.0
9,0,2000-01-10,10.0,2.0
10,0,2000-01-11,11.0,3.0
11,0,2000-01-12,12.0,4.0
12,0,2000-01-13,13.0,5.0


#### Fourier Terms

In [ ]:
full_df.columns

In [ ]:
with LogTime():
    full_df, added_features = bulk_add_fourier_features(
        full_df,
        ["timestamp_Month", "timestamp_Hour", "timestamp_Minute"],
        max_values=[12, 24, 60],
        n_fourier_terms=5,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 8 seconds, 805 milliseconds and 160 microseconds
Features Created: timestamp_Month_sin_1,timestamp_Month_sin_2,timestamp_Month_sin_3,timestamp_Month_sin_4,timestamp_Month_sin_5,timestamp_Month_cos_1,timestamp_Month_cos_2,timestamp_Month_cos_3,timestamp_Month_cos_4,timestamp_Month_cos_5,timestamp_Hour_sin_1,timestamp_Hour_sin_2,timestamp_Hour_sin_3,timestamp_Hour_sin_4,timestamp_Hour_sin_5,timestamp_Hour_cos_1,timestamp_Hour_cos_2,timestamp_Hour_cos_3,timestamp_Hour_cos_4,timestamp_Hour_cos_5,timestamp_Minute_sin_1,timestamp_Minute_sin_2,timestamp_Minute_sin_3,timestamp_Minute_sin_4,timestamp_Minute_sin_5,timestamp_Minute_cos_1,timestamp_Minute_cos_2,timestamp_Minute_cos_3,timestamp_Minute_cos_4,timestamp_Minute_cos_5


#### Plotting Fourier Terms

In [ ]:
plot_df = (
    full_df[["timestamp_Month", "timestamp_Month_sin_1"]]
    .drop_duplicates()
    .sort_values("timestamp_Month")
)
plot_df.columns = ["calendar", "fourier"]

plot_df = pd.concat([plot_df, plot_df, plot_df]).reset_index(drop=True)
# plot_df.reset_index(drop=True, inplace=True)

plot_df.reset_index(inplace=True)
plot_df["index"] += 1
plot_df = pd.melt(
    plot_df, id_vars="index", var_name="month", value_name="Representation"
)

In [ ]:
fig = px.line(plot_df, x="index", y="Representation", facet_row="month")
fig.update_layout(
    autosize=False,
    width=900,
    height=800,
    title_text="Step Function vs Continuous Function",
    title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
    titlefont={"size": 20},
    legend_title=None,

    xaxis=dict(
        title_text="Time",
    ),
)
fig.update_yaxes(matches=None)

In [ ]:
fig.update_xaxes(
    ticktext=np.arange(1, 13).tolist() * 3,
    tickvals=np.arange(len(plot_df)) + 1,
)
fig.show()

Saving the feature engineered file

In [ ]:
full_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 5000000 entries, 0 to 28655
Columns: 94 entries, timestamp to timestamp_Minute_cos_5
dtypes: datetime64[ns](1), float32(60), float64(8), int32(14), int64(2), object(9)
memory usage: 4.2 GB


In [ ]:
train_df = pd.read_parquet("df_train_feature_eng_2.parquet")
validation_df = pd.read_parquet("df_val_feature_eng_2.parquet")
test_df = pd.read_parquet("df_test_feature_eng_2.parquet")

#### Standardize Code to train and evaluate ML models

To standardize the pipeline we define three classes FeatureConfig, MissingValueConfig, ModelConfig and a wrapper class MLForecast over scikit-learn estimators (.fit and .predict) to make process smooth.

##### FeatureConfig

FeatureConfig defines the few key attributes and functions to process the data - continuous, categorical, and boolean columns require separate pre-processing before fed into ML model.

FeatureConfig holds
- mandatory date column with the date in the DataFrame.
- mandatory target column of the target in the DataFrame.
- original target if target has been transformed (like log-transformed, differenced and more).
- list of the continuous features, categorical and boolean features.
- index columns that are set as DataFrame index while preprocessing - typically is datetime and unique ID of time series as indices.
- exogenous features as a result of feature engineering process, such as lag and rolling features but also external sources such as temperature data.

In [ ]:
feat_config = FeatureConfig(
    date="timestamp",
    target="energy_consumption",
    continuous_features=[
        'visibility', 'windBearing', 'temperature', 'dewPoint', 'pressure', 'apparentTemperature', 'windSpeed', 'humidity',
        'energy_consumption_lag_1', 'energy_consumption_lag_2', 'energy_consumption_lag_3', 'energy_consumption_lag_4','energy_consumption_lag_5', 'energy_consumption_lag_46', 'energy_consumption_lag_47', 'energy_consumption_lag_48', 'energy_consumption_lag_49', 'energy_consumption_lag_50', 'energy_consumption_lag_334', 'energy_consumption_lag_335', 'energy_consumption_lag_336', 'energy_consumption_lag_337', 'energy_consumption_lag_338', 'energy_consumption_rolling_3_mean', 'energy_consumption_rolling_3_std', 'energy_consumption_rolling_6_mean', 'energy_consumption_rolling_6_std','energy_consumption_rolling_12_mean', 'energy_consumption_rolling_12_std', 'energy_consumption_rolling_48_mean', 'energy_consumption_rolling_48_std', 'energy_consumption_48_seasonal_rolling_3_mean', 'energy_consumption_48_seasonal_rolling_3_std', 'energy_consumption_336_seasonal_rolling_3_mean', 'energy_consumption_336_seasonal_rolling_3_std', 'energy_consumption_ewma_span_2880', 'energy_consumption_ewma_span_336', 'energy_consumption_ewma_span_48',
        'timestamp_Day', 'timestamp_Elapsed', 'timestamp_Month_sin_1', 'timestamp_Month_sin_2', 'timestamp_Month_sin_3', 'timestamp_Month_sin_4', 'timestamp_Month_sin_5', 'timestamp_Month_cos_1', 'timestamp_Month_cos_2', 'timestamp_Month_cos_3', 'timestamp_Month_cos_4', 'timestamp_Month_cos_5', 'timestamp_Hour_sin_1', 'timestamp_Hour_sin_2', 'timestamp_Hour_sin_3', 'timestamp_Hour_sin_4', 'timestamp_Hour_sin_5', 'timestamp_Hour_cos_1', 'timestamp_Hour_cos_2', 'timestamp_Hour_cos_3', 'timestamp_Hour_cos_4', 'timestamp_Hour_cos_5', 'timestamp_Minute_sin_1', 'timestamp_Minute_sin_2', 'timestamp_Minute_sin_3', 'timestamp_Minute_sin_4', 'timestamp_Minute_sin_5', 'timestamp_Minute_cos_1', 'timestamp_Minute_cos_2', 'timestamp_Minute_cos_3', 'timestamp_Minute_cos_4', 'timestamp_Minute_cos_5'
    ],

    categorical_features=[
        'holidays', 'precipType', 'icon', 'summary', 'timestamp_Month','timestamp_Quarter', 'timestamp_WeekDay', 'timestamp_Dayofweek','timestamp_Dayofyear', 'timestamp_Hour', 'timestamp_Minute',
    ],

    boolean_features=[
        'timestamp_Is_quarter_end', 'timestamp_Is_quarter_start', 'timestamp_Is_year_end', 'timestamp_Is_year_start', 'timestamp_Is_month_start',
    ],

    index_cols=["timestamp"],

    exogenous_features=[
        'holidays', 'precipType', 'icon', 'summary', 'visibility', 'windBearing', 'temperature', 'dewPoint', 'pressure', 'apparentTemperature', 'windSpeed', 'humidity',
    ],

)

In [ ]:
test_df.head(1)

,timestamp,LCLid,energy_consumption,timestamp_Week,frequency,series_length,stdorToU,Acorn,Acorn_grouped,holidays,...,timestamp_Minute_sin_1,timestamp_Minute_sin_2,timestamp_Minute_sin_3,timestamp_Minute_sin_4,timestamp_Minute_sin_5,timestamp_Minute_cos_1,timestamp_Minute_cos_2,timestamp_Minute_cos_3,timestamp_Minute_cos_4,timestamp_Minute_cos_5
36576,2014-02-01,MAC000026,0.178,5,30min,37872,Std,ACORN-D,Affluent,NO_HOLIDAY,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0


Take a sample from household

In [ ]:
sample_train_df = train_df.loc[train_df.LCLid == "MAC000026", :]

train_features, train_target, train_original_target = feat_config.get_X_y(
    sample_train_df, categorical=False, exogenous=False)

del sample_train_df

In [ ]:
sample_test_df = test_df.loc[test_df.LCLid == "MAC000026", :]

# Loading the Validation as test
test_features, test_target, test_original_target = feat_config.get_X_y(
    sample_test_df, categorical=False, exogenous=False)

del sample_test_df

In [ ]:
nc = train_features.isnull().sum()
nc[nc>0]

,0
energy_consumption_lag_5,246
energy_consumption_336_seasonal_rolling_3_std,6054
energy_consumption_rolling_48_mean,571
energy_consumption_lag_47,288
energy_consumption_336_seasonal_rolling_3_mean,6054
energy_consumption_lag_334,575
energy_consumption_lag_1,242
energy_consumption_lag_50,291
energy_consumption_lag_3,244
energy_consumption_rolling_12_mean,319


In [ ]:
nc = test_features.isnull().sum()
nc[nc>0]

,0
energy_consumption_336_seasonal_rolling_3_std,676
energy_consumption_336_seasonal_rolling_3_mean,676
energy_consumption_48_seasonal_rolling_3_mean,1488
energy_consumption_48_seasonal_rolling_3_std,1488


In [ ]:
test_features.head()

,timestamp_Minute_cos_2,energy_consumption_lag_5,timestamp_Minute_cos_1,energy_consumption_336_seasonal_rolling_3_std,timestamp_Minute_sin_4,energy_consumption_rolling_48_mean,timestamp_Month_cos_3,timestamp_Hour_cos_1,energy_consumption_lag_47,energy_consumption_336_seasonal_rolling_3_mean,...,energy_consumption_lag_48,timestamp_Month_cos_2,energy_consumption_lag_46,timestamp_Hour_sin_2,timestamp_Month_cos_4,timestamp_Month_cos_5,timestamp_Minute_sin_5,timestamp_Minute_cos_5,timestamp_Minute_sin_1,timestamp_Hour_sin_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-01-01 00:00:00,1.0,0.162,1.0,NaN,0.000000e+00,0.132646,6.123234e-17,1.000000,0.156,NaN,...,0.098,0.5,0.111,0.000000,-0.5,-0.866025,0.000000e+00,1.0,0.000000e+00,0.000000
2014-01-01 00:30:00,1.0,0.097,-1.0,NaN,-2.266215e-15,0.132771,6.123234e-17,1.000000,0.111,NaN,...,0.156,0.5,0.112,0.000000,-0.5,-0.866025,6.123234e-16,-1.0,5.665539e-16,0.000000
2014-01-01 01:00:00,1.0,0.126,1.0,NaN,0.000000e+00,0.132792,6.123234e-17,0.965926,0.112,NaN,...,0.111,0.5,0.153,0.500000,-0.5,-0.866025,0.000000e+00,1.0,0.000000e+00,0.965926
2014-01-01 01:30:00,1.0,0.140,-1.0,NaN,-2.266215e-15,0.132792,6.123234e-17,0.965926,0.153,NaN,...,0.112,0.5,0.101,0.500000,-0.5,-0.866025,6.123234e-16,-1.0,5.665539e-16,0.965926
2014-01-01 02:00:00,1.0,0.120,1.0,NaN,0.000000e+00,0.132875,6.123234e-17,0.866025,0.101,NaN,...,0.153,0.5,0.156,0.866025,-0.5,-0.866025,0.000000e+00,1.0,0.000000e+00,0.500000


##### MissingValueConfig

Some models handle the missing values and for other models we need to deal with them. We define the config class to set a few columns where we expect to fill the missing values. MissingValueConfig holds
- A list of columns that need to use backward fill strategy to fill missing values.
- A list of columns to use forward fill strategy to fill the missing values
- A list of columns that need to be filled with zeros.



In [ ]:
missing_value_config = MissingValueConfig(
	bfill_columns=["energy_consumption_lag_1", "energy_consumption_lag_2", "energy_consumption_lag_3", "energy_consumption_lag_4", "energy_consumption_lag_5", "energy_consumption_lag_46","energy_consumption_lag_47", "energy_consumption_lag_48","energy_consumption_lag_49", "energy_consumption_lag_50","energy_consumption_lag_334", "energy_consumption_lag_335",
	"energy_consumption_lag_336", "energy_consumption_lag_337",
	"energy_consumption_lag_338", "energy_consumption_rolling_3_mean",
	"energy_consumption_rolling_3_std", "energy_consumption_rolling_6_mean",
	"energy_consumption_rolling_6_std", "energy_consumption_rolling_12_mean",
	"energy_consumption_rolling_12_std", "energy_consumption_rolling_48_mean",
	"energy_consumption_rolling_48_std", "energy_consumption_48_seasonal_rolling_3_mean","energy_consumption_48_seasonal_rolling_3_std","energy_consumption_336_seasonal_rolling_3_mean",
	"energy_consumption_336_seasonal_rolling_3_std","energy_consumption_ewma__span_2880", "energy_consumption_ewma__span_336",
	"energy_consumption_ewma__span_48",
	],
		ffill_columns=[],
		zero_fill_columns=[],
)

In [ ]:
baseline_metrics_df = pd.read_pickle('single_step_backtesting_baseline_val_metrics_df.pkl')

ML model on sample data

In [ ]:
predictions_df = pd.concat([train_target, test_target])

metric_record = []
metric_record += (
	baseline_metrics_df.loc[baseline_metrics_df.LCLid == "MAC000026"]
	.drop(columns="LCLid")
	.to_dict(orient="records")
)

In [ ]:
metric_record

[{'Algorithm': 'Naive',
  'forecast_bias': 5.510752688172003e-05,
  'mae': 0.057190860349462365,
  'mase': 0.6775468215794397,
  'mse': 0.01658787380188179,
  'rmse': 0.12879391989485292},
 {'Algorithm': 'SeasonalNaive',
  'forecast_bias': 0.01569422049731183,
  'mae': 0.06504233864247312,
  'mase': 0.7705642045952455,
  'mse': 0.024458561031720488,
  'rmse': 0.15639233047601947}]

In [ ]:
from sklearn.preprocessing import StandardScaler

def evaluate_model(model_config, feature_config, missing_config, train_features, train_target, test_features, test_target):
    # Apply robust missing value handling to train_features before fitting the model
    processed_train_features = train_features.copy()
    for col in missing_config.bfill_columns:
        if col in processed_train_features.columns:
            processed_train_features[col] = processed_train_features[col].bfill().ffill().fillna(0)
    for col in missing_config.ffill_columns:
        if col in processed_train_features.columns:
            processed_train_features[col] = processed_train_features[col].ffill().bfill().fillna(0)
    for col in missing_config.zero_fill_columns:
        if col in processed_train_features.columns:
            processed_train_features[col] = processed_train_features[col].fillna(0)

    model_instance = model_config.model

    scaler_X, scaler_y = None, None
    if model_config.normalize:
        scaler_X = StandardScaler()
        processed_train_features = pd.DataFrame(scaler_X.fit_transform(processed_train_features),
                                                columns=processed_train_features.columns,
                                                index=processed_train_features.index)
        scaler_y = StandardScaler()
        train_target = pd.DataFrame(scaler_y.fit_transform(train_target),
                                     columns=train_target.columns,
                                     index=train_target.index)

    # Convert y to 1D array if needed by the model (e.g., for RandomForestRegressor, LightGBM)
    if hasattr(model_instance, 'fit') and 'y' in model_instance.fit.__code__.co_varnames and train_target.shape[1] == 1:
        model_instance.fit(processed_train_features, train_target.iloc[:, 0])
    else:
        model_instance.fit(processed_train_features, train_target)

    # Apply robust missing value handling to test_features before prediction
    processed_test_features = test_features.copy()
    for col in missing_config.bfill_columns:
        if col in processed_test_features.columns:
            processed_test_features[col] = processed_test_features[col].bfill().ffill().fillna(0)
    for col in missing_config.ffill_columns:
        if col in processed_test_features.columns:
            processed_test_features[col] = processed_test_features[col].ffill().bfill().fillna(0)
    for col in missing_config.zero_fill_columns:
        if col in processed_test_features.columns:
            processed_test_features[col] = processed_test_features[col].fillna(0)

    # Handle normalization for test data
    if model_config.normalize and scaler_X:
        processed_test_features = pd.DataFrame(scaler_X.transform(processed_test_features),
                                               columns=processed_test_features.columns,
                                               index=processed_test_features.index)

    # Predict
    y_pred_scaled = model_instance.predict(processed_test_features)
    y_pred = pd.Series(y_pred_scaled, index=test_features.index)

    # Inverse transform predictions if normalization was applied
    if model_config.normalize and scaler_y:
        # Squeeze to convert (n_samples, 1) array to 1D array for pd.Series constructor
        y_pred = pd.Series(scaler_y.inverse_transform(y_pred.to_frame()).squeeze(), index=y_pred.index, name=y_pred.name)

    metrics = calculate_metrics(y_pred, test_target, name=model_config.name)
    return y_pred, metrics

In [ ]:
from itertools import cycle

In [ ]:
def highlight_abs_min(s, props=''):
    return np.where(s == np.nanmin(np.abs(s.values)), props, '')

Linear Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRFRegressor
from lightgbm import LGBMRegressor

In [ ]:
# Correct missing_value_config to fix EWMA feature names
corrected_missing_value_config = MissingValueConfig(
    bfill_columns=[
        "energy_consumption_lag_1", "energy_consumption_lag_2","energy_consumption_lag_3", "energy_consumption_lag_4","energy_consumption_lag_5", "energy_consumption_lag_46","energy_consumption_lag_47", "energy_consumption_lag_48","energy_consumption_lag_49", "energy_consumption_lag_50","energy_consumption_lag_334", "energy_consumption_lag_335","energy_consumption_lag_336", "energy_consumption_lag_337","energy_consumption_lag_338", "energy_consumption_rolling_3_mean","energy_consumption_rolling_3_std", "energy_consumption_rolling_6_mean","energy_consumption_rolling_6_std", "energy_consumption_rolling_12_mean","energy_consumption_rolling_12_std","energy_consumption_rolling_48_mean","energy_consumption_rolling_48_std","energy_consumption_48_seasonal_rolling_3_mean","energy_consumption_48_seasonal_rolling_3_std","energy_consumption_336_seasonal_rolling_3_mean","energy_consumption_336_seasonal_rolling_3_std","energy_consumption_ewma_span_2880", "energy_consumption_ewma_span_336", "energy_consumption_ewma_span_48",
    ],
    ffill_columns=[],
    zero_fill_columns=[],
)

##### ModelConfig

ModelConfig class holds details regarding modeling process such as data normalization, filling missing values, categorical encoding and more. ModelConfig it holds
- Model (any scikit-learn estimator) and Model name
- normalize whether to apply StandardScaler
- fill_missing flag whether to fill  missing values before training or not
- encode_categorical whether to encode categorical columns as part of the fitting procedure
- categorical_encoder whether to use scikit-learn encoder



##### MLForecast

MLForecast is a wrapper around scikit-learn model, and uses different configurations to encapsulate the training and prediction functions.

The parameters available when initializing the model
- model_config instance of the class itself
- feature_config instance class
- missing_config instance class
- target_transformer instance of target transformers supporting fit, transform and inverse transform methods.


##### The fit function
The fit function is similar to the scikit-learn function but does more processing handling standardization, categorical encoding, and target transformation.


##### The predict function
The predict function handles inferencing, wraps around the predict function of scikit-learn estimator with additional functionality.


##### The feature_importance function
The feature_importance function retrieves the feature importance from the model if available. For linear models, it extracts the coefficients, while for tree-based models, it extracts the built-in importance and returns in sorted DataFrame.


##### The evaluate_model function
The evaluate_model is a helper function to evaluate different models and automate the process at scale. We use another utility function to calculate the metrics

Now that we have the baselines and a standard way to apply different models, we take a look at how different the models are.

Linear Regression

In [ ]:
model_config = ModelConfig(
    model=LinearRegression(),
    name="LinearRegression",
    normalize = True,
    fill_missing = False # Set fill_missing to False as we handle it explicitly
)

# Combine train_features and train_target to drop NaNs coherently
combined_train_data = pd.concat([train_features, train_target], axis=1)

# Drop rows where the target (energy_consumption) is NaN
# Assumes 'energy_consumption' is the name of the target column in train_target
initial_rows = combined_train_data.shape[0]
combined_train_data.dropna(subset=['energy_consumption'], inplace=True)
dropped_rows = initial_rows - combined_train_data.shape[0]
if dropped_rows > 0:
    print(f"Dropped {dropped_rows} rows from training data due to NaN in target.")

# Separate them back
cleaned_train_features = combined_train_data.drop(columns=['energy_consumption'])
cleaned_train_target = combined_train_data[['energy_consumption']]

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config = corrected_missing_value_config, # Pass the corrected config
		train_features = cleaned_train_features,
		train_target = cleaned_train_target,
		test_features = test_features,
		test_target = test_target,
	)
metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = predictions_df.join(y_pred)

Dropped 241 rows from training data due to NaN in target.
Time Elapsed: 331 milliseconds and 49 microseconds


In [ ]:
mae_str = f"{metrics['MAE']:.4f}" if metrics['MAE'] is not None else 'N/A'
mse_str = f"{metrics['MSE']:.4f}" if metrics['MSE'] is not None else 'N/A'
mase_str = f"{metrics['MASE']:.4f}" if metrics['MASE'] is not None else 'N/A'
bias_str = f"{metrics['Forecast Bias']:.4f}" if metrics['Forecast Bias'] is not None else 'N/A'

fig = plot_forecast(
    pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(
    fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

Ridge Regression (L2)

In [ ]:
model_config = ModelConfig(
	model=RidgeCV(),
	name="Ridge Regression",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=corrected_missing_value_config, #missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

Time Elapsed: 437 milliseconds and 620 microseconds


In [ ]:
metrics

{'Algorithm': 'Ridge Regression',
 'MAE': np.float64(0.05191081703989873),
 'MSE': np.float64(0.01249350934986251),
 'MASE': None,
 'Forecast Bias': np.float64(-3.308649710808108),
 'Time Elapsed': 0.4376204013824463}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

Lasso Regression

In [ ]:
model_config = ModelConfig(
	model=RidgeCV(),
	name="Lasso Regression",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=corrected_missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

Time Elapsed: 380 milliseconds and 677 microseconds


In [ ]:
metrics

{'Algorithm': 'Lasso Regression',
 'MAE': np.float64(0.05191081703989873),
 'MSE': np.float64(0.01249350934986251),
 'MASE': None,
 'Forecast Bias': np.float64(-3.308649710808108),
 'Time Elapsed': 0.3806772232055664}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

Decision Tree

In [ ]:
model_config = ModelConfig(
	model=DecisionTreeRegressor(max_depth=4, random_state=42),
	name="Decision Tree",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

Time Elapsed: 797 milliseconds and 145 microseconds


In [ ]:
metrics

{'Algorithm': 'Decision Tree',
 'MAE': np.float64(0.05323133919071977),
 'MSE': np.float64(0.01264228042947409),
 'MASE': None,
 'Forecast Bias': np.float64(7.3667745629850385),
 'Time Elapsed': 0.7971446514129639}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

Bagging and Boosting Trees

Random Forest

In [ ]:
model_config = ModelConfig(
	model=RandomForestRegressor(max_depth=4, random_state=42),
	name="Random Forest",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().



Time Elapsed: 41 seconds, 739 milliseconds and 600 microseconds


In [ ]:
metrics

{'Algorithm': 'Random Forest',
 'MAE': np.float64(0.053254961697598115),
 'MSE': np.float64(0.012244417052267803),
 'MASE': None,
 'Forecast Bias': np.float64(7.879201122862487),
 'Time Elapsed': 41.73960018157959}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

XGBoost Random Forest

In [ ]:
model_config = ModelConfig(
	model=XGBRFRegressor(max_depth=4, random_state=42),
	name="XGB Random Forest",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: FutureWarning:

`XGBRFRegressor` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.



Time Elapsed: 1 second, 329 milliseconds and 841 microseconds


In [ ]:
metrics

{'Algorithm': 'XGB Random Forest',
 'MAE': np.float64(0.05286944388436861),
 'MSE': np.float64(0.012092457785182295),
 'MASE': None,
 'Forecast Bias': np.float64(7.247220184530219),
 'Time Elapsed': 1.3298406600952148}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

LightGBM

In [ ]:
model_config = ModelConfig(
	model=LGBMRegressor(random_state=42),
	name="Light GBM",
	normalize=True,
	fill_missing=False # Set fill_missing to False as we handle it explicitly
)

with LogTime() as timer:
	y_pred, metrics = evaluate_model(
		model_config,
		feat_config,
		missing_config=missing_value_config,
		train_features=cleaned_train_features,
		train_target=cleaned_train_target,
		test_features=test_features,
		test_target=test_target,
	)

metrics["Time Elapsed"] = timer.elapsed
metric_record.append(metrics)
pred_df = pred_df.join(y_pred)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007666 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8157
[LightGBM] [Info] Number of data points in the train set: 34847, number of used features: 60
[LightGBM] [Info] Start training from score 0.257698
Time Elapsed: 1 second, 603 milliseconds and 809 microseconds


In [ ]:
metrics

{'Algorithm': 'Light GBM',
 'MAE': np.float64(0.048555242946792074),
 'MSE': np.float64(0.01142033763688776),
 'MASE': None,
 'Forecast Bias': np.float64(0.9670928761843149),
 'Time Elapsed': 1.603808879852295}

In [ ]:
fig = plot_forecast(pred_df, forecast_columns=[model_config.name], forecast_display_names=[model_config.name])
fig = format_plot(fig, title=f"{model_config.name}: MAE: {mae_str} | MSE: {mse_str} | MASE: {mase_str} | Bias: {bias_str}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

Summary

In [ ]:
formatted = pd.DataFrame(metric_record).style.format(
	{"MAE": "{:.4f}",
	 "MSE": "{:.4f}",
	 "MASE": "{:.4f}",
	 "Forecast Bias": "{:.2f}%"})

formatted.highlight_min(color='lightgreen', subset= ["MAE", "MSE","MASE"]).apply(
    highlight_abs_min,
    props='color:black;background-color:lightgreen', axis=0,
    subset=['Forecast Bias'])

,Algorithm,forecast_bias,mae,mase,mse,rmse,MAE,MSE,MASE,Forecast Bias,Time Elapsed
0,Naive,0.000055,0.057191,0.677547,0.016588,0.128794,nan,nan,nan,nan%,nan
1,SeasonalNaive,0.015694,0.065042,0.770564,0.024459,0.156392,nan,nan,nan,nan%,nan
2,LinearRegression,nan,nan,nan,nan,nan,0.0519,0.0125,nan,-3.32%,0.331049
3,Ridge Regression,nan,nan,nan,nan,nan,0.0519,0.0125,nan,-3.31%,0.437620
4,Lasso Regression,nan,nan,nan,nan,nan,0.0519,0.0125,nan,-3.31%,0.380677
5,Decision Tree,nan,nan,nan,nan,nan,0.0532,0.0126,nan,7.37%,0.797145
6,Random Forest,nan,nan,nan,nan,nan,0.0533,0.0122,nan,7.88%,41.739600
7,XGB Random Forest,nan,nan,nan,nan,nan,0.0529,0.0121,nan,7.25%,1.329841
8,Light GBM,nan,nan,nan,nan,nan,0.0486,0.0114,nan,0.97%,1.603809


Runnning ML Forecast for all consumers

In [ ]:
lcl_ids = sorted(train_df.LCLid.unique())

models_to_run = [
	ModelConfig(model=LassoCV(), name="Lasso Regression", normalize=True, fill_missing=True),
	ModelConfig(model=RandomForestRegressor(random_state=42, max_depth=4),name="Random Forest", normalize=False,fill_missing=False,), # Changed from XGBRFRegressor to RandomForestRegressor
	ModelConfig(model=LGBMRegressor(random_state=42),name="LightGBM",normalize=False,fill_missing=False,
	)
]

all_preds = []
all_metrics = []

In [ ]:
all_preds = []
all_metrics = []

# Filter out rows where 'energy_consumption' is missing from train_df
initial_train_rows = train_df.shape[0]
train_df.dropna(subset=['energy_consumption'], inplace=True)
dropped_train_rows = initial_train_rows - train_df.shape[0]
if dropped_train_rows > 0:
    print(f"Dropped {dropped_train_rows} rows from global train_df due to NaN in 'energy_consumption'.")

# Filter out rows where 'energy_consumption' is missing from test_df
initial_test_rows = test_df.shape[0]
test_df.dropna(subset=['energy_consumption'], inplace=True)
dropped_test_rows = initial_test_rows - test_df.shape[0]
if dropped_test_rows > 0:
    print(f"Dropped {dropped_test_rows} rows from global test_df due to NaN in 'energy_consumption'.")

with LogTime() as timer:
	for lcl_id in tqdm(lcl_ids):
		# Extract features and targets for the current LCLid
		X_train, y_train, _ = feat_config.get_X_y(
			train_df.loc[train_df.LCLid == lcl_id, :], categorical=False, exogenous=False)
		X_test, y_test, _ = feat_config.get_X_y(
			test_df.loc[test_df.LCLid == lcl_id, :], categorical=False, exogenous=False
		)

		# Crucial check for empty dataframes before proceeding
		# If any of these are empty, we cannot train or predict for this LCLid.
		if X_train.empty or y_train.empty or X_test.empty or y_test.empty:
			print(f"Skipping LCLid {lcl_id} due to empty training or testing data after extraction.")
			continue

		for model_config in models_to_run:
			model_config = model_config.clone()
			with warnings.catch_warnings():
				warnings.simplefilter("ignore")
				y_pred, metrics = evaluate_model(
					model_config,
					feat_config,
					missing_config=corrected_missing_value_config, # Use the corrected missing_value_value_config
					train_features=X_train,
					train_target=y_train,
					test_features=X_test,
					test_target=y_test,
				)
			y_pred.name = "predictions"
			y_pred = y_pred.to_frame()
			y_pred["LCLid"] = lcl_id
			y_pred["Algorithm"] = model_config.name
			metrics["LCLid"] = lcl_id
			metrics["Algorithm"] = model_config.name
			y_pred["energy_consumption"] = y_test.values
			all_preds.append(y_pred)
			all_metrics.append(metrics)

In [ ]:
pred_df = pd.concat(all_preds)
pred_df.head()

,predictions,LCLid,Algorithm,energy_consumption
timestamp,,,,
2014-02-01 00:00:00,0.152389,MAC000026,Lasso Regression,0.178
2014-02-01 00:30:00,0.139119,MAC000026,Lasso Regression,0.092
2014-02-01 01:00:00,0.088697,MAC000026,Lasso Regression,0.115
2014-02-01 01:30:00,0.105112,MAC000026,Lasso Regression,0.135
2014-02-01 02:00:00,0.123536,MAC000026,Lasso Regression,0.143


In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.head()

,Algorithm,MAE,MSE,MASE,Forecast Bias,LCLid
0,Lasso Regression,0.117099,0.040026,None,-5.200081,MAC000026
1,Random Forest,0.117396,0.040114,None,-3.712085,MAC000026
2,LightGBM,0.114287,0.037918,None,-4.549742,MAC000026
3,Lasso Regression,0.100815,0.031688,None,3.388959,MAC000052
4,Random Forest,0.090061,0.031089,None,-3.487404,MAC000052


In [ ]:
baseline_aggregate_metrics_df = pd.read_pickle("single_step_backtesting_agg_metric_val_df.pkl")
baseline_aggregate_metrics_df.head()

,MAE,MSE,meanMASE,Forecast Bias
Naive,0.105377,0.052115,1.110506,0.003839
Seasonal Naive,0.173180,0.119894,1.853693,0.149545


In [ ]:
metrics = baseline_aggregate_metrics_df.reset_index().rename(columns={"index":"Algorithm"}).to_dict(orient="records")

In [ ]:
for model_config in models_to_run:
		pred_mask = pred_df.Algorithm==model_config.name
		metric_mask = metrics_df.Algorithm==model_config.name
		metrics.append({
		"Algorithm": model_config.name,
		"MAE": mae(pred_df.loc[pred_mask,"energy_consumption"], pred_df.loc[pred_mask,"predictions"]),
		"MSE": mse(pred_df.loc[pred_mask,"energy_consumption"], pred_df.loc[pred_mask,"predictions"]),
		"meanMASE": metrics_df.loc[metric_mask, "MASE"].mean(),
		"Forecast Bias": forecast_bias_aggregate(pred_df.loc[pred_mask,"energy_consumption"], pred_df.loc[pred_mask,"predictions"])
})

In [ ]:
agg_metrics_df = pd.DataFrame(metrics)

In [ ]:
def highlight_abs_min(s, props=''):
    return np.where(s == np.nanmin(np.abs(s.values)), props, '')

agg_metrics_df.style.highlight_min(color='lightgreen', subset= ["MAE", "MSE","meanMASE"]).apply(
    highlight_abs_min,
    props='color:black;background-color:lightgreen', axis=0,
    subset=['Forecast Bias'])

,Algorithm,MAE,MSE,meanMASE,Forecast Bias
0,Naive,0.105377,0.052115,1.110506,0.003839
1,Seasonal Naive,0.173180,0.119894,1.853693,0.149545
2,Lasso Regression,0.098389,0.038128,nan,-1.551117
3,Random Forest,0.097714,0.038557,nan,-0.826721
4,LightGBM,0.094901,0.035513,nan,2.810228


In [ ]:
pred_df.to_pickle("ml_single_step_predictions_val_df.pkl")
metrics_df.to_pickle("ml_single_step_metrics_val_df.pkl")
agg_metrics_df.to_pickle("ml_single_step_agg_metrics_val_df.pkl")

In [ ]:
files.download("ml_single_step_predictions_val_df.pkl")
files.download("ml_single_step_metrics_val_df.pkl")
files.download("ml_single_step_agg_metrics_val_df.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>